# Shadow Cell Validation 001 — Copy-on-Write Functional Isolation

Independent kill test. Fresh synthetic data, fresh base checkpoint per seed, fresh formal seeds. No Native CLM M2/M3 checkpoint or dataset is used. Running the formal cell deliberately consumes formal seeds `95211/95212/95213`.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/shadow-cell-validation-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/experiments/shadow-cell-validation-001-copy-on-write-functional-isolation'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', BRANCH])
    run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing GITHUB_TOKEN'
assert torch.cuda.is_available(), 'CUDA required for canonical Shadow Cell validation'
assert torch.cuda.device_count() >= 2, f'Canonical runner requires two GPUs, found {torch.cuda.device_count()}'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Verify the frozen independent protocol before consuming any formal seed.
protocol_path = Path('research/validations/shadow-cell-validation-001-copy-on-write-functional-isolation/protocol.json')
protocol = json.loads(protocol_path.read_text())
assert protocol['status'] == 'FROZEN_UNRUN'
assert protocol['independent_of_native_clm_m2_chain'] is True
assert protocol['fresh_evidence']['formal_seeds'] == [95211, 95212, 95213]
assert protocol['B_adaptation']['certificate_projection'] is False
assert protocol['B_adaptation']['historical_replay_in_weight_training'] is False
print(json.dumps({
    'world': protocol['world'],
    'formal_seeds': protocol['fresh_evidence']['formal_seeds'],
    'maturity_grid': protocol['maturity_grid'],
    'thresholds': protocol['thresholds'],
}, indent=2))

In [ ]:
# FORMAL RUN: fresh base training + matched direct/shadow adaptation for all three untouched formal seeds.
run([
    sys.executable, 'scripts/research/run_shadow_cell_validation_001.py',
    '--phase', 'formal',
    '--devices', '0,1',
    '--output-dir', OUT,
])
decision = json.loads((OUT / 'decision.json').read_text())
print(json.dumps({
    'classification': decision['classification'],
    'scientific_decision': decision['scientific_decision'],
    'seeds': decision['seeds'],
    'protocol_sha256': decision['protocol_sha256'],
}, indent=2))

In [ ]:
# Compact causal readout before publication.
for result in decision['seed_results']:
    print('seed', result['seed'])
    print(' base A acc:', result['base_metrics']['A']['accuracy'])
    print(' parent A/B:', result['parent']['top1_share_A'], result['parent']['top1_share_B'])
    print(' direct B gain:', result['direct_tx']['B_gain'])
    print(' gate AUC:', result['gate']['heldout_auc'])
    print(' primary conditional:', result['primary_conditional'])
    print(' HV:', result['hypervolume'])
    print(' shuffled control:', result['causal_control'])
    print(' identity:', result['identity'])

In [ ]:
# HF-first checkpoint publication, then lightweight Git evidence publication.
run([
    sys.executable, 'scripts/research/publish_shadow_cell_validation_001.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
])
print('Published Shadow Cell Validation 001:', decision['classification'])